# Broadcast -> .srt transcription with WhisperX (Kaggle T4 GPU)

WhisperX = faster-whisper + wav2vec2 word-level alignment + pyannote speaker
diarization on a Darija / Arabic / French broadcast clip.

**Setup:**
- Settings -> Accelerator -> **GPU T4 x2** (or P100)
- Internet **On** (first run downloads model weights ~3 GB + pyannote models)
- The repo is cloned automatically — just run all cells
- To use your own `.mp3`, upload via **Add Data** -> Upload and set `CLIP` manually
- **For diarization:** a HuggingFace read token (see cell 3)

**Two modes:**
1. Plain transcription (no diarization) — run cells 1-8
2. With speaker diarization — also run cells 9-12

## 1. Install dependencies

In [ ]:
!pip -q install "whisperx>=3.8.0" "pyannote.audio>=3.1.0" "transformers>=4.46.0" "peft>=0.14.0"

## 2. Clone the repo & load pipeline code

The notebook auto-clones the transcription repo so you get the latest pipeline code
and a sample broadcast clip. No manual dataset upload needed.

In [ ]:
import os, sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/haca-transcription")
if not REPO_ROOT.exists():
    !git clone --depth 1 https://github.com/MarTCM/haca-transcription.git "{REPO_ROOT}"
else:
    print("[repo] already cloned")

sys.path.insert(0, str(REPO_ROOT / "src"))
import transcribe_whisperx as wx
from srt_writer import write_srt
print("[repo]", REPO_ROOT)

## 3. HuggingFace token (required for diarization)

Speaker diarization uses **pyannote.audio** gated models. You need:
1. A HuggingFace account at [huggingface.co](https://huggingface.co)
2. Accept terms at:
   - https://huggingface.co/pyannote/speaker-diarization-community-1
   - https://huggingface.co/pyannote/segmentation-3.0
3. A **read token** from https://huggingface.co/settings/tokens

Paste it below (or leave empty to skip diarization).

In [ ]:
HF_TOKEN = ""  # <-- paste your HuggingFace read token here, or leave empty

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set")
else:
    print("HF_TOKEN empty — diarization will be skipped")

## 4. Choose a clip

By default, uses the trimmed broadcast clip from the repo. To transcribe your own
audio, upload it via **Add Data** -> Upload, then set `CLIP` to its path below.

In [ ]:
import glob

# --- set this to your uploaded file, or leave the default ---
CLIP = str(REPO_ROOT / "samples" / "mobachara_ma3akom_trimmed.mp3")

# Also list user-uploaded media under /kaggle/input for convenience.
uploaded = sorted(
    p for p in glob.glob("/kaggle/input/**/*", recursive=True)
    if os.path.splitext(p)[1].lower() in wx.MEDIA_EXTS
)
if uploaded:
    print("uploaded media files:")
    for p in uploaded:
        print("  ", p)

assert os.path.exists(CLIP), f"clip not found: {CLIP}"
print("\nusing CLIP:", CLIP)

## 5. Load WhisperX model on GPU

`large-v3-turbo` in `float16`. First run downloads the weights (~3 GB).
For best Darija quality, flip the toggle below to enable the anaszil LoRA adapter
(see also `kaggle_darija_lora.ipynb`). The LoRA now includes word-level timestamps
so speaker diarization works on Arabic chunks too (set `HF_TOKEN` and `--diarize`).

In [ ]:
# --- toggle: set to True to route Arabic chunks through the anaszil LoRA ---
USE_DARIJA_LORA = False

model = wx.load_model("large-v3-turbo", device="cuda", compute_type="float16")
lora_pipe = None
if USE_DARIJA_LORA:
    lora_pipe = wx._load_darija_lora(device="cuda")

## 6. Transcribe (per-chunk language detection, no diarization)

`lang="auto"` detects the language per ~25 s chunk (allow-list `ar,fr,en`),
so a code-switched broadcast transcribes natively. Alignment is attempted
per language; Arabic uses a community wav2vec2 model, French uses the default.
If alignment fails for a language, Whisper's native timestamps are kept.

In [ ]:
segments = wx.transcribe_file(
    CLIP, model,
    lang="auto", allowed=("ar", "fr", "en"),
    max_chunk_s=25.0, batch_size=8, beam_size=5,
    device="cuda",
    darija_lora=USE_DARIJA_LORA, lora_pipe=lora_pipe,
)

print(f"\n{len(segments)} cues total")
for s in segments[:15]:
    print(f"  [{s['start']:6.1f}-{s['end']:6.1f}] {s['text']}")

from collections import Counter
print(f"\nlang mix: {Counter(s['text'].split(']')[0].strip('[') if s['text'].startswith('[') else 'no-speaker' for s in segments)}")

## 7. Write the .srt

In [ ]:
out_name = os.path.splitext(os.path.basename(CLIP))[0]
out_path_no_diar = write_srt(segments, f"/kaggle/working/{out_name}_whisperx.srt")
print("wrote", out_path_no_diar)
print(out_path_no_diar.read_text(encoding="utf-8")[:2000])

## 8. Verify the .srt is valid

In [ ]:
import re
BLOCK = re.compile(
    r"(\d+)\n"
    r"(\d\d:\d\d:\d\d,\d\d\d) --> (\d\d:\d\d:\d\d,\d\d\d)\n"
    r"(.+?)(?=\n\n|\Z)", re.DOTALL
)
text = out_path_no_diar.read_text(encoding="utf-8")
cues = BLOCK.findall(text)
print(f"parsed {len(cues)} cues; first:", cues[0] if cues else None)
assert len(cues) == len(segments), "cue count mismatch"
print("SRT format OK")

---
## 9. (Optional) Run with speaker diarization

This runs the full WhisperX pipeline including pyannote speaker diarization.
Requires `HF_TOKEN` to be set in cell 3. The first run downloads pyannote
models (~200 MB). Speaker labels appear as `[SPEAKER_00]` prefixes in the
SRT text.

In [ ]:
if not HF_TOKEN:
    print("Skipping diarization — HF_TOKEN not set (see cell 3)")
else:
    segs_diar = wx.transcribe_file(
        CLIP, model,
        lang="auto", allowed=("ar", "fr", "en"),
        max_chunk_s=25.0, batch_size=8, beam_size=5,
        diarize=True, hf_token=HF_TOKEN,
        min_speakers=None, max_speakers=None,
        device="cuda",
        darija_lora=USE_DARIJA_LORA, lora_pipe=lora_pipe,
    )

    print(f"\n{len(segs_diar)} cues total (with speakers)")
    for s in segs_diar[:15]:
        print(f"  [{s['start']:6.1f}-{s['end']:6.1f}] {s['text']}")

    # Speaker statistics
    import re as _re
    spk_counter = Counter()
    for s in segs_diar:
        m = _re.match(r'^\[(SPEAKER_\d+)\]', s['text'])
        if m:
            spk_counter[m.group(1)] += 1
    print(f"\nSpeaker cue counts: {dict(spk_counter)}")
    if spk_counter:
        total = sum(spk_counter.values())
        for spk, count in spk_counter.most_common():
            print(f"  {spk}: {count} cues ({count/total*100:.0f}%)")

## 10. Write speaker-labeled .srt

In [ ]:
if HF_TOKEN:
    out_path_diar = write_srt(
        segs_diar,
        f"/kaggle/working/{out_name}_whisperx_diarized.srt",
    )
    print("wrote", out_path_diar)
    print(out_path_diar.read_text(encoding="utf-8")[:2000])

## 11. Download & summary

Both `.srt` files are in `/kaggle/working/`. Download them from the Kaggle
**Output** panel (`/kaggle/working/...srt`).

| File | Content |
|---|---|
| `{clip}_whisperx.srt` | Plain transcription (per-chunk code-switching) |
| `{clip}_whisperx_diarized.srt` | Transcription + [SPEAKER_XX] labels |

**Tip:** If you uploaded multiple clips, change `CLIP` in cell 4 and re-run
cells 6-10 to transcribe each one.

In [ ]:
print("Output files in /kaggle/working/:")
for p in sorted(glob.glob("/kaggle/working/*.srt")):
    size = os.path.getsize(p)
    print(f"  {os.path.basename(p)}  ({size:,} bytes)")